**Linda Zier**

**ST 554 ~ Final Project**

**04/30/2026**

# Goal

The goal of this project was to use Spark to fit a machine learning model and apply it to streaming data. All project files were housed in a GitHub repository, including a Jupyter notebook and a Python producer script. In the notebook, I fit an elastic net regression model using PySpark's MLlib module to predict power consumption in Zone 3 of Tetouan City from weather and time variables. I then simulated a data stream by writing a separate Python script that periodically wrote batches of data to a monitored folder. As data arrived in the stream, I used the fitted model to generate predictions and wrote the results out to the console.

### Data

The data used in this project is modified from the UCI Machine Learning Repository and is available at https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv. The dataset contains measurements from Tetouan City relating power consumption across three zones to factors such as temperature, humidity, wind speed, diffuse solar flows, and time of day. I used this dataset to fit my model, treating Power Zone 3 consumption as the response variable. A separate streaming dataset (power_streaming_data.csv) was used to simulate incoming data. My producer script repeatedly sampled from this file and wrote batches to a monitored folder where the fitted model generated predictions on the arriving data.

# Fitting the Model

I created a Jupyter notebook for the model fitting part and the streaming part below. I read the data into a standard pandas data frame using the pd.read_csv() function and converted this to a spark data frame.

In [1]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, VectorAssembler, Binarizer, \
                               OneHotEncoder, PCA
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql.functions import col
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder.getOrCreate()

# read in data/power_ml_data.csv as pandas dataframe
powerDF=pd.read_csv("data/power_ml_data.csv")

#convert to spark dataframe
powerDF=spark.createDataFrame(powerDF)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/30 14:33:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/30 14:33:35 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/30 14:33:35 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/30 14:33:35 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/04/30 14:33:35 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
26/04/30 14:33:35 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.


### Creating the Pipeline

I fit an elastic net model using cross validation with the steps below. The transformations used an MLlib function that was put into a pipeline.

*   I used an SQL transformer to cast the hour variable as DoubleType.
*   I binarized the Hour column based on the column being less than 6.5 or not (night vs day essentially).
*   The month column was one-hot encoded.
*   I ran a Principle Component Analysis (PCA) on the Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, and Diffuse_Flows columns. I did this by:

        - First using a VectorAssembler() to place these variables in a single column for use with the PCA() estimator  
        
        - Then applying a PCA transformer for use in our pipeline using two principle components.
        
        
*   I renamed our response variable Power_Zone_3 as label.

*   I used VectorAssembler() to put my predictors into a features vector. The predictors were:

    – Two fitted PCA features
    
    – Binary Hour variable
    
    – Power_Zone_1
    
    – Power_Zone_2
    
    – Month indicator variables
    
I then built my pipeline for the transformations.
    


In [2]:
# check the data types
powerDF.printSchema()

# cast the hour as double since it is a long
sql = SQLTransformer(statement = '''
                     SELECT *, 
                     CAST(Hour AS DOUBLE) AS HourD FROM __THIS__
                     ''')

# binarize night vs day
binarizer = Binarizer(threshold=6.5, inputCol="HourD", outputCol="Hour_bin")

# one hot encode month
ohe = OneHotEncoder(inputCols=["Month"], outputCols=["Month_ohe"])

#VectorAssembler to bundle features together for pca
pca_assembler = VectorAssembler(
    inputCols=["Temperature", "Humidity", "Wind_Speed", 
               "General_Diffuse_Flows", "Diffuse_Flows"],
    outputCol="pca_input")

# pca with 2 components
pca = PCA(k=2, inputCol="pca_input", outputCol="pca_features")

# response variable to label
sql_label= SQLTransformer(statement = '''
                          SELECT *,
                          Power_Zone_3 AS label FROM __THIS__
                          ''')
# assemble final features
assembler = VectorAssembler(
    inputCols=["pca_features", "Hour_bin", "Power_Zone_1", 
               "Power_Zone_2","Month_ohe"],
    outputCol="features")

print("TRANSFORMATIONS COMPLETE")   
                     

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)

TRANSFORMATIONS COMPLETE


In [3]:
from pyspark.ml import Pipeline

#build pipeline
pipeline= Pipeline(stages = [sql, binarizer, ohe, pca_assembler, 
                             pca, sql_label, assembler])
fittedPipeline = pipeline.fit(powerDF)
transformedDF=fittedPipeline.transform(powerDF)

print("PIPELINE COMPLETE")

PIPELINE COMPLETE


### Fitting an Elastic Net Model

Next I used the CrossValidator and LinearRegression functions to fit an elastic net model. I searched over multiple combinations of regularization parameters, which control the strength of the penalty, and elastic net parameters, which control the mix between Lasso and Ridge. The model was fit using 5-fold cross validation with root mean square error (RMSE) as the criteria: I trained 5 separate models (one per fold) and averaged their RMSEs to evaluate each parameter combination. I then reported the optimal tuning parameter values and the CV error, which is the RMSE from the best model.


In [4]:

# setting up elastic net model
lr= LinearRegression(elasticNetParam=0.5)


#  grid for the regParam and elasticNetParam
paramGrid= ParamGridBuilder() \
    .addGrid(lr.regParam,[0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam,[ 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()

# 5-fold CV with rmse evaluator
cv=CrossValidator(estimator=lr,
                   estimatorParamMaps = paramGrid,  
                   evaluator = RegressionEvaluator(metricName= 'rmse'),
                   numFolds=5)

# fit the model
cvModel=cv.fit(transformedDF)

# Report the optimal values chosen for the tuning parameters
print("Optimal regularization parameter: ", cvModel.bestModel.getRegParam())
print("Optimal elastic net parameter: ", cvModel.bestModel.getElasticNetParam())

# report RMSE errors - if you want to see all 11 x 11 = 121 of them
# print("RMSE errors= ", cvModel.avgMetrics)

# report lowest RMSE
print("CV RMSE: ", min(cvModel.avgMetrics))


26/04/30 14:33:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/30 14:33:48 WARN Instrumentation: [6cb9a577] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 14:33:50 WARN Instrumentation: [6cb9a577] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/30 14:33:51 WARN Instrumentation: [f35e8adc] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 14:33:52 WARN Instrumentation: [f35e8adc] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/30 14:33:52 WARN Instrumentation: [f0f63d1c] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 14:33:52 WARN Instrumentation: [f0f63d1c] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
2

Optimal regularization parameter:  0.05
Optimal elastic net parameter:  0.05
CV RMSE:  2147.6875170325898


### Training Set Evaluation
I reported the training set RMSE using our fitted model as a transformer and
evaluating on the entire training set. I then created a residual column (label - prediction) and printed the data frame with these residuals. I also printed a summary table as a sanity check to ensure the mean was near zero and the standard deviation was reasonable.

In [5]:
# report training set RMSE by using fitted model as a transformer
predictions = cvModel.transform(transformedDF)
trainRMSE = RegressionEvaluator(metricName= 'rmse').evaluate(predictions)
print("Training RMSE=", trainRMSE)

# create residual column and display results
print("Training set residuals and summary:")
predictions = predictions.withColumn("residual", col("label") - col("prediction"))
predictions.select("label", "prediction", "residual").show()

#sanity check on residual distribution
predictions.select("label", "prediction", "residual") \
    .summary("mean", "stddev", "min", "max") \
    .show()

Training RMSE= 2147.097295248907
Training set residuals and summary:
+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|20240.96386|20879.670678519215|-638.7068185192147|
|20131.08434|18660.129333474768|1470.9550065252333|
|19668.43373| 18204.65111302254|1463.7826169774598|
|18899.27711|17590.569940826674|1308.7071691733254|
|18442.40964|16997.220954537923|1445.1886854620789|
|18130.12048|16517.614600106695| 1612.505879893306|
|17945.06024| 16093.18540963103| 1851.874830368968|
|17459.27711|15722.637868825801| 1736.639241174198|
|17025.54217|15270.995577219088|1754.5465927809128|
|16794.21687|14938.301657159245| 1855.915212840755|
|16638.07229| 14652.38338677063|1985.6889032293693|
|16395.18072|14414.904162932282|1980.2765570677184|
|16117.59036|14082.850243233228|2034.7401167667722|
| 15822.6506|13624.855902916494|2197.7946970835073|
|15672.28916|13450.339719713815|2221.9494402861

# Handling Streaming Data
I downloaded the streaming source file from https://www4.stat.ncsu.edu/~online/datasets/power_streaming_data.csv
and stored it in my final_project/data directory.  This file served as my source for random sampling in my producer script.
### Reading a Stream
I read in a stream in the form of .csv files. I created a folder (using mkdir in terminal) called streaming_data to serve as the monitored directory. The schema was set to that of the original data since that is what my incoming data would have the same structure and a header was assumed to be present.

In [6]:
# set the schema to that of the original data
stream_schema=powerDF.schema
print(stream_schema)

# set up readstream with a header
streamDF = spark.readStream.schema(stream_schema).option("header", True)\
           .csv("streaming_data")

#showing current working directory
#import os
#os.getcwd()

StructType([StructField('Temperature', DoubleType(), True), StructField('Humidity', DoubleType(), True), StructField('Wind_Speed', DoubleType(), True), StructField('General_Diffuse_Flows', DoubleType(), True), StructField('Diffuse_Flows', DoubleType(), True), StructField('Power_Zone_1', DoubleType(), True), StructField('Power_Zone_2', DoubleType(), True), StructField('Power_Zone_3', DoubleType(), True), StructField('Month', LongType(), True), StructField('Hour', LongType(), True)])



### Transform/Aggregation Step

In this code block I used the fitted model as a transformer to obtain predictions from the incoming data stream. First I created a residual column (label - prediction) as done in the previous section, returning only the label, prediction, and residual columns. Then, using a second transformation on the original stream, I renamed the response variable Power_Zone_3 to label, keeping all other columns intact. Finally I joined the two transformations together on the label variable using an inner join.

In [7]:
# ---Transformation 1:---

# apply transformer to the data stream
streamPredictions = cvModel.transform(fittedPipeline.transform(streamDF))

#add residual column = label - predictions
streamResiduals=streamPredictions.withColumn("residual",col("label")- col("prediction")) \
                                  .select("label","prediction", "residual")
                                              
# ---Transformation 2:---

# rename response to
streamLabeled=sql_label.transform(streamDF)

# --- Inner Join ---
streamJoined = streamResiduals.join(streamLabeled, on="label", how="inner")



### Writing Step
I wrote the joined stream to the console using append output mode, which outputs only newly arriving rows with each batch. I started the query and allowed it to run for 20 seconds before terminating it with query.stop().

In [11]:
# Write stream to console in append mode and start query
#query = streamJoined.writeStream.outputMode("append").format("console")

query = streamJoined.writeStream \
                    .outputMode("append") \
                    .format("console") \
                    .start()


26/04/30 14:41:39 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-fd5e7e1e-7916-467e-8bd9-0cec2b234b97. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/30 14:41:39 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|    11827.2|15697.773474313159| -3870.573474313158|      20.37|    79.5|     0.075|                 7.09|        6.014| 24565.82781| 13528.06653|     11827.2|    6|   6|
|17638.55422| 19264.19922981994|-1625.6450098199384|       17.8|   65.35|      4.92|                222.8|        192.3| 35210.12658|  20600.6079| 17638.55422|    1|  13|
|25100.58577| 29402.61579716694|  -4302.03002716

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|27842.00837|27575.810295432228|   266.198074567772|      23.74|   53.09|     4.915|                0.099|        0.137| 31798.00664|     21600.0| 27842.00837|    7|   1|
|12072.02881| 9848.182570820485| 2223.8462391795147|      10.51|    71.2|     0.082|                0.059|        0.167| 27406.84411| 23362.99478| 12072.02881|   12|  23|
|16145.97839|17606.938190804267|-1460.9598008042

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|14991.29724|13733.750337960962|1257.5469020390374|      24.48|   55.35|     0.286|                464.9|        125.3| 33808.14159| 18984.19958| 14991.29724|    9|  10|
|11800.12158| 13072.22426573127| -1272.10268573127|      25.64|   36.98|      4.92|                565.8|         87.3| 35108.27133| 21726.97095| 11800.12158|   10|  10|
|12376.61538| 15294.35445360476| -2917.73907360476|  

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|20369.72308|15530.420375874452|  4839.302704125548|      25.67|   65.61|     4.925|                862.0|        54.88| 29912.58278| 16877.33888| 20369.72308|    6|  14|
|18020.40486| 17397.96096612282|  622.4438938771782|      21.85|   58.09|     0.071|                843.0|        52.94| 34308.19672| 21685.44892| 18020.40486|    5|  14|
|9218.313253| 14510.84089528807| -5292.527642288

-------------------------------------------
Batch: 4
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|31848.70293|  30542.6687203063| 1306.0342096936984|      32.78|   33.61|      4.91|                745.0|         95.4| 41079.06977| 30941.77215| 31848.70293|    7|  11|
|9254.261705|  7055.98730297016| 2198.2744020298405|      11.93|   48.23|     0.077|                0.066|        0.085| 21000.76046| 17140.22706| 9254.261705|   12|   4|
|13820.46987| 15695.92158978595|-1875.4517197859

-------------------------------------------
Batch: 5
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|20363.81538| 20720.24074802945|-356.4253680294496|      22.06|    75.8|     0.069|                164.5|        132.2| 35704.37086| 20406.23701| 20363.81538|    6|  18|
|7571.668667| 7223.680852574932| 347.9878144250679|       8.67|    86.9|     0.084|                47.95|        34.46| 24292.01521| 18377.41639| 7571.668667|   12|   8|
|25383.76569|26459.523592905596|-1075.757902905596|  

-------------------------------------------
Batch: 6
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|19637.68997| 22761.75927080936| -3124.069300809362|      22.85|   61.79|     0.081|                0.066|        0.063|  47397.1116| 28792.53112| 19637.68997|   10|  19|
|15848.72727|17233.330943218378|-1384.6036732183784|      18.33|   64.84|     0.084|                616.1|         93.3| 30734.46717| 21786.96538| 15848.72727|    4|  10|
|25014.03015| 24856.43679081277| 157.59335918722

In [10]:
# stop querying
query.stop()

### the other commands below were used when repeatedly testing
#for s in spark.streams.active:
#    s.stop()
    
#for f in os.listdir("streaming_data"):
#    os.remove(f"streaming_data/{f}")